# LLM-as-Judge Evaluation Harness
## Use LLMs to Score LLMs — Building Evaluation Harnesses
⏱ ~12 min

**LLM-as-Judge** is the most practical way to evaluate generative AI pipelines at scale. Instead of hand-labelling hundreds of outputs, you ask a second LLM to score each response against a structured rubric. By the end of this session you will understand *why* automated evaluation matters, *how* to design a scoring rubric an LLM can apply consistently, and *how* to build a full judge harness that produces reproducible, aggregated metrics.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why automate evaluation? LLM judges vs alternatives |
| 2 | **RAG Pipeline Under Test** — Build the system we will evaluate |
| 3 | **Test Dataset** — Define questions, answers, and reference ground truth |
| 4 | **Judge Rubric** — Design a structured scoring schema with Pydantic |
| 5 | **Judge Harness** — Run the judge and collect per-answer scores |
| 6 | **Aggregation & Reporting** — Compute dimension averages, identify failures |
| 7 | **Bias & Calibration** — Positional bias, self-enhancement, mitigation |
| ★ | **Advanced Techniques** — Reference-free scoring, pairwise ranking |

---

### Prerequisites
- Python 3.10+ (a `.venv` with requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Zheng, L., Chiang, W.-L., et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* NeurIPS 2023. https://arxiv.org/abs/2306.05685  
> Liu, Y., Iter, D., et al. (2023). *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment.* https://arxiv.org/abs/2303.16634  
> Chang, Y., et al. (2024). *A Survey on Evaluation of Large Language Models.* ACM TIST. https://arxiv.org/abs/2307.03109

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "chromadb",
            "python-dotenv",
            "pydantic",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import json
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed.\n"
    "Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True (ends with ...{key[-4:]})")

---

## Part 1 — Why Automate Evaluation?

---

### The Evaluation Problem

You built a RAG pipeline and it *seems* to work. But how do you know?

- **Human evaluation** is accurate but expensive and slow — a team of annotators scoring 500 outputs takes days
- **Reference-based metrics** (ROUGE, BLEU, BERTScore) compare to a gold answer token-by-token — but a response can be semantically correct with completely different wording and score poorly
- **LLM judges** score the *meaning* of a response against a rubric — fast, scalable, and strongly correlated with human judgements (Zheng et al. 2023)

---

### Evaluation Method Comparison

| Method | Speed | Cost | Measures meaning | Requires gold labels | Use when |
|--------|-------|------|-----------------|---------------------|----------|
| **Human annotation** | Slow | High | Yes | No | Final benchmark, legal/compliance |
| **ROUGE / BLEU** | Fast | Free | No (n-gram overlap) | Yes | Translation, summarization |
| **BERTScore** | Fast | Free | Partial | Yes | Requires reference |
| **RAGAS** | Fast | Low | Partial | Partial | RAG-specific statistical metrics |
| **LLM-as-Judge** | Fast | Low-medium | Yes | No | General QA, RAG, chatbots |
| **Pairwise ranking** | Medium | Medium | Yes | No | Comparing two systems |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Judge LLM** | A second model whose only job is to score system outputs |
| **Rubric** | A structured set of criteria and scoring levels the judge applies |
| **Relevance** | Does the answer address the question that was asked? |
| **Faithfulness** | Is every claim in the answer grounded in the retrieved context? |
| **Completeness** | Does the answer cover all key points from the reference? |
| **Positional bias** | Judge's tendency to prefer the first answer when comparing two |
| **Self-enhancement bias** | A model scoring its own outputs higher than others |
| **Calibration** | Alignment between judge scores and human ground truth scores |

### Judge Pipeline Architecture

```
EVALUATION PIPELINE
────────────────────────────────────────────────────────────

  Test Dataset (questions + references)
         │
         ▼
  ┌──────────────┐
  │  RAG Pipeline │  generates candidate answers
  │  (under test) │  for each test question
  └──────┬───────┘
         │  answer + retrieved context
         ▼
  ┌──────────────────────────────────────┐
  │           Judge Prompt               │
  │  ┌──────────────────────────────┐   │
  │  │  Question                    │   │
  │  │  Reference answer            │   │
  │  │  Retrieved context           │   │
  │  │  Candidate answer            │   │
  │  │  Rubric (Rel / Faith / Comp) │   │
  │  └──────────────────────────────┘   │
  └──────┬───────────────────────────────┘
         │
         ▼
  ┌──────────────┐
  │  Judge LLM   │  outputs structured JSON
  │ (gpt-4o-mini)│  via with_structured_output
  └──────┬───────┘
         │  {relevance, faithfulness,
         │   completeness, reasoning}
         ▼
  ┌──────────────┐
  │  Aggregator  │  average per dimension,
  └──────┬───────┘  flag low-scoring items
         │
         ▼
  Evaluation Report (metrics table + failure list)
```

> **Source**: Architecture inspired by Zheng et al. (2023) MT-Bench and G-Eval (Liu et al. 2023).

---

## Part 2 — RAG Pipeline Under Test

---

Before we can evaluate anything, we need a system to evaluate. We will build a small RAG pipeline over a LangGraph/RAG knowledge base — the same subject matter as examples 17, 27, and 30 in this series.

This section deliberately keeps the RAG pipeline simple so we can focus on the evaluation mechanics. The judge harness in Parts 4-6 works with any RAG pipeline — swap out this one for your own.

In [ ]:
# ─── 2-A: Imports and model setup ────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pydantic import BaseModel, Field

# Separate model instances: one for RAG generation, one for judging.
# Using the same model for both is valid but creates self-enhancement risk.
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Models ready.")

In [ ]:
# ─── 2-B: Knowledge base documents ───────────────────────────────────────────
# These six documents are the ground truth the RAG pipeline can access.
# The judge will later check whether answers are FAITHFUL to these documents.

DOCUMENTS = [
    Document(
        page_content=(
            "LangGraph is a library for building stateful, multi-actor applications "
            "with LLMs, built on top of LangChain. It models agent workflows as "
            "directed graphs where nodes are functions and edges are transitions."
        ),
        metadata={"source": "langgraph-docs"},
    ),
    Document(
        page_content=(
            "RAG (Retrieval-Augmented Generation) combines a retriever with a "
            "generative model to ground answers in source documents. The retriever "
            "fetches the top-k relevant chunks; the generator produces the answer."
        ),
        metadata={"source": "rag-overview"},
    ),
    Document(
        page_content=(
            "Corrective RAG (CRAG) grades each retrieved document for relevance "
            "and rewrites the query if any document scores irrelevant. It improves "
            "retrieval precision without changing the generation model."
        ),
        metadata={"source": "crag-paper"},
    ),
    Document(
        page_content=(
            "Self-RAG generates special reflection tokens (Retrieve, ISREL, ISSUP, "
            "ISUSE) to decide whether retrieval is needed before fetching any "
            "documents. This makes retrieval selective rather than always-on."
        ),
        metadata={"source": "self-rag-paper"},
    ),
    Document(
        page_content=(
            "Human-in-the-loop (HITL) checkpointing pauses LangGraph execution at "
            "an interrupt node and waits for human approval before resuming. "
            "State is persisted to a checkpointer (SQLite or Postgres)."
        ),
        metadata={"source": "hitl-docs"},
    ),
    Document(
        page_content=(
            "Agentic RAG combines a ReAct-style agent loop with retrieval tools. "
            "The agent decides when to retrieve, what to query, and whether to "
            "iterate again — rather than doing a single retrieve-then-generate pass."
        ),
        metadata={"source": "agentic-rag-docs"},
    ),
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:70]}...")

In [ ]:
# ─── 2-C: Build the RAG chain ─────────────────────────────────────────────────
store = Chroma.from_documents(DOCUMENTS, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer using ONLY the provided context. "
            "Be concise — 1-3 sentences. If the context does not contain the answer, "
            "say 'I don't have that information.'",
        ),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)


def run_rag(question: str) -> tuple[str, list[Document]]:
    """Run the RAG pipeline. Returns (answer, retrieved_docs)."""
    docs = retriever.invoke(question)
    ctx = "\n\n".join(d.page_content for d in docs)
    answer = (rag_prompt | rag_llm | StrOutputParser()).invoke(
        {"context": ctx, "question": question}
    )
    return answer, docs


# Smoke test
test_answer, test_docs = run_rag("What is LangGraph?")
print("RAG pipeline ready.")
print(f"Smoke test answer: {test_answer}")
print(f"Retrieved {len(test_docs)} docs: {[d.metadata['source'] for d in test_docs]}")

---

## Part 3 — Test Dataset

---

A good evaluation dataset covers:
1. **In-scope questions** the pipeline should answer correctly
2. **Borderline questions** that require combining multiple chunks
3. **Out-of-scope questions** where the model should decline gracefully

Each item has:
- `question` — what the user asks
- `reference` — the ideal answer (ground truth, written by a human)

The `reference` field is optional for some judge configurations but improves **completeness** scoring accuracy.

In [ ]:
# ─── 3-A: Define the test dataset ─────────────────────────────────────────────
TEST_SET = [
    {
        "question": "What is LangGraph?",
        "reference": (
            "LangGraph is a library for building stateful, multi-actor LLM applications "
            "that models workflows as directed graphs with nodes and edges."
        ),
    },
    {
        "question": "How does Corrective RAG handle irrelevant documents?",
        "reference": (
            "CRAG grades each retrieved document for relevance and rewrites the query "
            "when any document scores irrelevant, improving retrieval precision."
        ),
    },
    {
        "question": "What are reflection tokens in Self-RAG?",
        "reference": (
            "Self-RAG generates reflection tokens (Retrieve, ISREL, ISSUP, ISUSE) "
            "to decide whether retrieval is needed before fetching documents."
        ),
    },
    {
        "question": "What happens at a human-in-the-loop interrupt node?",
        "reference": (
            "Graph execution pauses at the interrupt node, state is persisted to "
            "a checkpointer, and execution resumes after human approval."
        ),
    },
    {
        "question": "How is Agentic RAG different from standard RAG?",
        "reference": (
            "Agentic RAG uses a ReAct loop so the agent decides when and what to "
            "retrieve and whether to iterate, rather than a single retrieve-generate pass."
        ),
    },
    {
        "question": "What is the capital of France?",
        "reference": "Paris — but this is out of scope for this knowledge base.",
    },
]

print(f"Test dataset: {len(TEST_SET)} questions ({len(TEST_SET) - 1} in-scope, 1 out-of-scope)")

In [ ]:
# ─── 3-B: Generate RAG answers for all test questions ─────────────────────────
results = []
for item in TEST_SET:
    answer, docs = run_rag(item["question"])
    results.append(
        {
            **item,
            "answer": answer,
            "context": "\n".join(d.page_content for d in docs),
            "sources": [d.metadata["source"] for d in docs],
        }
    )

print("=" * 60)
for r in results:
    print(f"Q: {r['question']}")
    print(f"A: {r['answer']}")
    print(f"   [sources: {', '.join(r['sources'])}]")
    print()

---

## Part 4 — Judge Rubric Design

---

### Why Structured Output?

A plain-text judge response like *"This answer is pretty good, 7/10"* is unprocessable. We need:
1. **Parseable scores** for aggregation (integers, not natural language)
2. **Per-dimension scores** so we can diagnose *which* dimension fails
3. **Reasoning text** so we can understand why a score was given (and spot judge errors)

LangChain's `with_structured_output` forces the model to emit valid JSON that matches a Pydantic schema — no fragile string parsing required.

---

### Scoring Rubric — 0-3 Scale

| Score | Relevance | Faithfulness | Completeness |
|-------|-----------|-------------|---------------|
| **0** | Completely off-topic | Contradicts the context | Empty or nonsensical |
| **1** | Tangentially related | Unsupported claims present | Misses most key points |
| **2** | Mostly on-topic | Mostly grounded, minor gaps | Covers most key points |
| **3** | Directly answers the question | Every claim is in the context | All key points covered |

> **Why 0-3 and not 1-10?** Coarser scales produce more consistent scores across judge runs. Zheng et al. (2023) found that 1-10 scales exhibit high variance; a 0-3 scale forces the judge to make cleaner categorical distinctions.

In [ ]:
# ─── 4-A: Define the Pydantic schema for judge output ─────────────────────────


class JudgeScore(BaseModel):
    relevance: int = Field(
        ...,
        ge=0,
        le=3,
        description="0=off-topic 1=tangential 2=mostly relevant 3=fully addresses the question",
    )
    faithfulness: int = Field(
        ...,
        ge=0,
        le=3,
        description="0=contradicts context 1=unsupported claims 2=mostly grounded 3=fully grounded",
    )
    completeness: int = Field(
        ...,
        ge=0,
        le=3,
        description="0=empty 1=misses most points 2=covers most points 3=covers all key points",
    )
    reasoning: str = Field(
        ...,
        description="One or two sentences explaining the scores. Be specific about what is missing or wrong.",
    )


# with_structured_output wraps the LLM so it always returns a JudgeScore object,
# not an AIMessage. The model is prompted internally to emit valid JSON.
judge = judge_llm.with_structured_output(JudgeScore)

print("Judge schema:")
for field_name, field in JudgeScore.model_fields.items():
    print(f"  {field_name}: {field.description}")

In [ ]:
# ─── 4-B: Write the judge prompt ──────────────────────────────────────────────
# The judge prompt is the most important design decision.
# Key rules:
#   1. Give the rubric explicitly — don't ask the judge to invent criteria
#   2. Provide the retrieved context (not just the answer) for faithfulness scoring
#   3. Provide the reference answer for completeness scoring
#   4. Ask for reasoning before scores to encourage chain-of-thought

JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator assessing the quality of a RAG system's answer.
Score the answer on three dimensions using a 0-3 scale.

=== INPUT ===
Question: {question}
Reference answer: {reference}
Retrieved context:
{context}

System answer: {answer}

=== RUBRIC ===
Relevance (0-3): Does the answer directly address the question?
  0 = completely off-topic
  1 = tangentially related
  2 = mostly on-topic
  3 = directly and fully addresses the question

Faithfulness (0-3): Is the answer grounded in the retrieved context?
  0 = contradicts the context or makes up facts
  1 = contains unsupported claims
  2 = mostly grounded, minor unsupported details
  3 = every claim is supported by the retrieved context

Completeness (0-3): Does the answer cover all key points from the reference?
  0 = empty or nonsensical
  1 = misses most key points
  2 = covers most key points
  3 = covers all key points from the reference answer

Provide reasoning FIRST, then scores.\
"""

# Verify the prompt fills correctly with one example
sample = results[0]
filled = JUDGE_PROMPT_TEMPLATE.format(**sample)
print(f"Prompt character count: {len(filled)}")
print("First 400 chars of filled prompt:")
print(filled[:400])

---

## Part 5 — Running the Judge Harness

---

The harness loops through every test result, calls the judge once per answer, and collects `JudgeScore` objects. Two things to notice:

- The judge receives `question`, `reference`, `context`, and `answer` — all four matter
- We store the full `JudgeScore` object so we can slice scores any way we want in Part 6

This cell will make **N API calls** (one per test item). With 6 items and `gpt-4o-mini` it costs well under $0.01.

In [ ]:
# ─── 5-A: Run the judge on every result ───────────────────────────────────────
scored = []

for r in results:
    prompt_text = JUDGE_PROMPT_TEMPLATE.format(**r)
    score: JudgeScore = judge.invoke(prompt_text)
    scored.append({**r, "score": score})
    print(f"Q: {r['question'][:60]}")
    print(
        f"   Relevance={score.relevance}  "
        f"Faithfulness={score.faithfulness}  "
        f"Completeness={score.completeness}"
    )
    print(f"   Reasoning: {score.reasoning}")
    print()

In [ ]:
# ─── 5-B: Inspect the structured output object ────────────────────────────────
# with_structured_output returns a real Pydantic model — you can access fields
# as attributes and validate them with standard Pydantic tools.

first_score = scored[0]["score"]
print("Type:", type(first_score))
print("Attributes:")
print(f"  .relevance     = {first_score.relevance}")
print(f"  .faithfulness  = {first_score.faithfulness}")
print(f"  .completeness  = {first_score.completeness}")
print(f"  .reasoning     = {first_score.reasoning}")
print()
print("JSON export (model_dump):")
print(json.dumps(first_score.model_dump(), indent=2))

---

## Part 6 — Aggregation and Reporting

---

Individual scores are useful for debugging. Aggregate metrics tell you whether the *system* is good enough to ship — and which dimension to fix first.

Common thresholds (set these based on your use-case):
- **Faithfulness < 2.5**: hallucination risk — fix retrieval or the generation prompt before deploying
- **Relevance < 2.0**: retrieval is returning the wrong documents — investigate `k`, chunk size, or query rewriting
- **Completeness < 2.0**: answers are too terse — check the generation prompt and `k`

In [ ]:
# ─── 6-A: Compute aggregate statistics ────────────────────────────────────────
n = len(scored)
dims = ["relevance", "faithfulness", "completeness"]

stats: dict[str, dict[str, float]] = {}
for dim in dims:
    vals = [getattr(r["score"], dim) for r in scored]
    stats[dim] = {
        "avg": sum(vals) / n,
        "min": min(vals),
        "max": max(vals),
    }

overall_avg = sum(stats[d]["avg"] for d in dims) / len(dims)

print(f"{'Dimension':<20} {'Avg':>6} {'Min':>6} {'Max':>6} {'/ 3':>5}")
print("-" * 44)
for dim in dims:
    s = stats[dim]
    bar = "█" * int(s["avg"] * 6)
    print(f"{dim.capitalize():<20} {s['avg']:>6.2f} {s['min']:>6} {s['max']:>6}   {bar}")
print("-" * 44)
print(f"{'Overall':<20} {overall_avg:>6.2f}")

# Threshold warnings
print()
for dim in dims:
    avg = stats[dim]["avg"]
    if avg < 2.0:
        print(f"WARNING: {dim} avg {avg:.2f} < 2.0 — investigate before deploying")

In [ ]:
# ─── 6-B: Identify low-scoring items for targeted debugging ───────────────────
FAIL_THRESHOLD = 2  # scores below this are flagged

failures = [
    r
    for r in scored
    if any(getattr(r["score"], d) < FAIL_THRESHOLD for d in dims)
]

print(f"Items with at least one dimension below {FAIL_THRESHOLD}: {len(failures)} / {n}")
print()

for r in failures:
    s = r["score"]
    failed_dims = [d for d in dims if getattr(s, d) < FAIL_THRESHOLD]
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:120]}...")
    print(f"   Failed dimensions: {', '.join(failed_dims)}")
    print(f"   Scores: rel={s.relevance} faith={s.faithfulness} comp={s.completeness}")
    print(f"   Reasoning: {s.reasoning}")
    print()

In [ ]:
# ─── 6-C: Full evaluation report as a table ───────────────────────────────────
header = f"{'#':<3} {'Question':<45} {'Rel':>4} {'Fai':>4} {'Com':>4} {'Avg':>5}"
print(header)
print("-" * len(header))

for i, r in enumerate(scored, 1):
    s = r["score"]
    avg = (s.relevance + s.faithfulness + s.completeness) / 3
    flag = "  <--" if avg < 2.0 else ""
    print(
        f"{i:<3} {r['question'][:44]:<45} "
        f"{s.relevance:>4} {s.faithfulness:>4} {s.completeness:>4} {avg:>5.2f}{flag}"
    )

---

## Part 7 — Bias and Calibration

---

LLM judges are powerful but imperfect. Three bias types you must understand before using judge scores to make deployment decisions:

---

### Bias Types and Mitigations

| Bias | What it is | Mitigation |
|------|-----------|------------|
| **Positional bias** | In pairwise scoring, the judge prefers the first response regardless of quality | Swap A/B order and average both runs |
| **Self-enhancement bias** | A model scores its own outputs higher than a different model's outputs | Use a different model as judge than the one being evaluated |
| **Verbosity bias** | Longer answers score higher even when they are less accurate | Include length-normalisation instructions in the rubric |
| **Sycophancy bias** | Judge inflates scores when the answer is confident or formal | Randomise phrasing style across test items |
| **Score range compression** | Judge avoids extreme scores (0 or 3); most scores land in 1-2 | Provide explicit examples of each score level in the rubric |

---

### Calibration Check

The simplest calibration test: write **deliberately bad answers** and verify the judge scores them low. If the judge awards a fabricated answer a 3 on faithfulness, your rubric or prompt needs work.

In [ ]:
# ─── 7-A: Calibration probe — score known-bad answers ─────────────────────────
# These deliberately broken answers let us check whether the judge's scores make sense.

calibration_cases = [
    {
        "label": "Fabricated (hallucinated) answer",
        "question": "What is LangGraph?",
        "reference": "LangGraph is a library for building stateful LLM applications modelled as directed graphs.",
        "context": DOCUMENTS[0].page_content,
        "answer": "LangGraph is a TypeScript framework for building REST APIs with automatic OpenAPI generation.",
    },
    {
        "label": "Off-topic answer",
        "question": "What is LangGraph?",
        "reference": "LangGraph is a library for building stateful LLM applications modelled as directed graphs.",
        "context": DOCUMENTS[0].page_content,
        "answer": "The weather in Paris today is partly cloudy with a high of 22 degrees.",
    },
    {
        "label": "Perfect answer",
        "question": "What is LangGraph?",
        "reference": "LangGraph is a library for building stateful LLM applications modelled as directed graphs.",
        "context": DOCUMENTS[0].page_content,
        "answer": (
            "LangGraph is a library built on LangChain for building stateful, multi-actor "
            "applications with LLMs, where workflows are modelled as directed graphs with "
            "nodes (functions) and edges (transitions)."
        ),
    },
]

print("Calibration Results:")
print("=" * 60)
for case in calibration_cases:
    prompt_text = JUDGE_PROMPT_TEMPLATE.format(
        question=case["question"],
        reference=case["reference"],
        context=case["context"],
        answer=case["answer"],
    )
    score: JudgeScore = judge.invoke(prompt_text)
    avg = (score.relevance + score.faithfulness + score.completeness) / 3
    print(f"\nCase: {case['label']}")
    print(f"Answer: {case['answer'][:80]}...")
    print(
        f"Scores: rel={score.relevance} faith={score.faithfulness} comp={score.completeness} avg={avg:.2f}"
    )
    print(f"Reasoning: {score.reasoning}")

---

## Part 8 ★ — Advanced Techniques (Bonus)

---

### Reference-Free Scoring

The harness above uses a `reference` answer for completeness scoring. In many real scenarios you don't have reference answers — you're evaluating a new pipeline with no labelled data. Reference-free scoring drops the `reference` field and asks the judge to evaluate answer quality against the retrieved context alone:

```python
# Faithfulness + relevance only — no reference needed
REFERENCE_FREE_PROMPT = """
Question: {question}
Retrieved context: {context}
System answer: {answer}

Score faithfulness (is every claim in the answer supported by the context?) and
relevance (does the answer address the question?) on a 0-3 scale.
"""
```

---

### Pairwise Ranking

Instead of absolute scores, ask the judge to pick which of two answers is better. This is the MT-Bench approach (Zheng et al. 2023) and is more robust to score compression:

```python
class PairwiseResult(BaseModel):
    winner: str = Field(..., description="'A' or 'B' or 'tie'")
    reasoning: str

# Always run both orderings (A vs B AND B vs A) to cancel positional bias.
```

---

### G-Eval Style Chain-of-Thought

G-Eval (Liu et al. 2023) improves alignment with human judgements by asking the model to first generate evaluation *steps* before producing scores:

```
Step 1: Read the question and identify what a complete answer requires.
Step 2: Read the retrieved context and list the relevant facts.
Step 3: Compare the system answer against the facts.
Step 4: Assign scores.
```

Add this as a `chain_of_thought: str` field to your Pydantic schema and include it in the prompt.

In [ ]:
# ─── 8-A: Add a conciseness dimension to the schema ───────────────────────────
# This is a worked example of extending JudgeScore — a preview of Exercise 1.


class ExtendedJudgeScore(BaseModel):
    relevance: int = Field(..., ge=0, le=3, description="Addresses the question directly")
    faithfulness: int = Field(..., ge=0, le=3, description="Grounded in retrieved context")
    completeness: int = Field(..., ge=0, le=3, description="Covers all key points")
    conciseness: int = Field(
        ...,
        ge=0,
        le=3,
        description=(
            "0=very verbose/padded 1=somewhat wordy 2=mostly concise "
            "3=tight and direct with no unnecessary words"
        ),
    )
    reasoning: str = Field(..., description="One sentence explaining all four scores")


extended_judge = judge_llm.with_structured_output(ExtendedJudgeScore)

EXTENDED_PROMPT = JUDGE_PROMPT_TEMPLATE + (
    "\n\nConciseness (0-3): Is the answer direct with no padding?\n"
    "  0 = very verbose, contains unnecessary filler\n"
    "  1 = somewhat wordy\n"
    "  2 = mostly concise\n"
    "  3 = tight and direct, every word earns its place"
)

# Score the first result with the extended schema
r0 = results[0]
ext_score: ExtendedJudgeScore = extended_judge.invoke(EXTENDED_PROMPT.format(**r0))

print(f"Q: {r0['question']}")
print(f"A: {r0['answer']}")
print(
    f"Extended scores: rel={ext_score.relevance} faith={ext_score.faithfulness} "
    f"comp={ext_score.completeness} concise={ext_score.conciseness}"
)
print(f"Reasoning: {ext_score.reasoning}")

---

## Exercises

---

### Exercise 1 — Add a Conciseness Dimension

The `ExtendedJudgeScore` in cell 8-A already shows the pattern. Your task:
1. Update `EXTENDED_PROMPT` to include the conciseness rubric
2. Re-run all 6 test items using `extended_judge`
3. Add `conciseness` to the aggregation table in Part 6
4. Which answers are high quality but verbose?

---

### Exercise 2 — Break Faithfulness

Change `search_kwargs={"k": 2}` to `search_kwargs={"k": 0}` so the RAG pipeline retrieves nothing (empty context). Re-run cells 3-B, 5-A, and 6-A.

**Expected result:** Faithfulness scores should collapse toward 0 because the model has no context to ground its answers in — it will hallucinate or say it doesn't know.

**Question to answer:** Does Relevance also drop? Why or why not?

---

### Exercise 3 — Compare Two Models

Swap the `rag_llm` model from `gpt-4o-mini` to any other available model. Re-run the full pipeline and compare the aggregate scores.

**Tip:** Run Part 3-B and Parts 5-6 but keep the judge model fixed (`gpt-4o-mini`). That way the evaluation standard is constant and the only variable is the system under test.

In [ ]:
# ── Exercise 1 Starter ─────────────────────────────────────────────────────────
# Extend the judge schema and prompt, then re-run the full harness.


class MyExtendedScore(BaseModel):
    relevance: int = Field(..., ge=0, le=3)
    faithfulness: int = Field(..., ge=0, le=3)
    completeness: int = Field(..., ge=0, le=3)
    # TODO: add conciseness field here
    reasoning: str


MY_EXTENDED_PROMPT = JUDGE_PROMPT_TEMPLATE + """

# TODO: add conciseness rubric section here
"""

my_extended_judge = judge_llm.with_structured_output(MyExtendedScore)

# TODO: loop through results and score with my_extended_judge
# TODO: print an aggregation table including conciseness

In [ ]:
# ── Exercise 2 Starter ─────────────────────────────────────────────────────────
# Break retrieval and observe faithfulness collapse.

# TODO: rebuild the retriever with k=0
# broken_retriever = store.as_retriever(search_kwargs={"k": 0})

# TODO: re-run run_rag with the broken retriever
# TODO: re-run the judge harness
# TODO: compare aggregated faithfulness before and after

In [ ]:
# ── Exercise 3 Starter ─────────────────────────────────────────────────────────
# Swap the RAG model and compare judge scores.

# TODO: create alt_rag_llm with a different model
# alt_rag_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# TODO: rebuild run_rag using alt_rag_llm
# TODO: re-run cells 3-B, 5-A, 6-A
# TODO: print a side-by-side comparison of avg scores

---

## What's Next?

You now have a working LLM-as-Judge harness. Here is where to go deeper:

### Improve the pipeline being evaluated
- **Example 17 — Corrective RAG** ([`../17-corrective-rag/`](../17-corrective-rag/)): add a relevance grader node before generation and watch Faithfulness scores improve
- **Example 27 — Self-RAG** ([`../27-self-rag/`](../27-self-rag/)): selective retrieval via reflection tokens — does retrieval on-demand score better than always-on?
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): compare agent-loop RAG vs standard RAG with the same judge harness

### Use statistical metrics alongside judge scores
- **Example 16 — RAG Evaluation with RAGAS** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): RAGAS computes Faithfulness and Answer Relevancy using statistical decomposition — compare its results to your judge scores on the same test set

### Scale the harness
- **Async evaluation**: replace `judge.invoke()` with `await judge.ainvoke()` in an `asyncio.gather()` loop — 10x faster on large test sets
- **Versioned runs**: log each evaluation run with a timestamp + model name to track regression over time
- **LangSmith**: enable tracing (`LANGSMITH_TRACING=true`) to see the full prompt + score for every judge call in a dashboard

### Further reading
- Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* https://arxiv.org/abs/2306.05685
- Liu et al. (2023). *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment.* https://arxiv.org/abs/2303.16634
- Chang et al. (2024). *A Survey on Evaluation of Large Language Models.* https://arxiv.org/abs/2307.03109
- LangChain structured output docs: https://python.langchain.com/docs/concepts/structured_outputs/

---

*Workshop complete. The next step is example 16 — apply RAGAS statistical metrics to the same RAG pipeline and compare.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

---

### Exercise 1 Answer — Add a Conciseness Dimension

**What to look for:** `gpt-4o-mini` answers tend to be short (1-3 sentences as prompted) so conciseness should score high (2-3) for most items. If you remove the "Be concise" instruction from `rag_prompt`, you will see conciseness scores drop — a useful confirmation that the dimension is responding to real signal.

**Gotcha:** Completeness and Conciseness trade off. An answer that covers all key points in 4 sentences may score 3 on completeness and 2 on conciseness, while a one-sentence answer scores 3 on conciseness and 1 on completeness. Neither is wrong — they reflect different quality goals.

In [ ]:
# Sample solution for Exercise 1


class ConciseJudgeScore(BaseModel):
    relevance: int = Field(..., ge=0, le=3)
    faithfulness: int = Field(..., ge=0, le=3)
    completeness: int = Field(..., ge=0, le=3)
    conciseness: int = Field(
        ...,
        ge=0,
        le=3,
        description="0=very verbose 1=wordy 2=mostly concise 3=tight and direct",
    )
    reasoning: str


CONCISE_PROMPT = JUDGE_PROMPT_TEMPLATE + (
    "\n\nConciseness (0-3): Is the answer direct with no padding?\n"
    "  0 = very verbose with unnecessary filler\n"
    "  1 = somewhat wordy\n"
    "  2 = mostly concise\n"
    "  3 = tight and direct, every word earns its place\n"
)

concise_judge = judge_llm.with_structured_output(ConciseJudgeScore)
concise_scored = []

for r in results:
    score = concise_judge.invoke(CONCISE_PROMPT.format(**r))
    concise_scored.append({**r, "score": score})

# Aggregation table
extended_dims = ["relevance", "faithfulness", "completeness", "conciseness"]
print(f"{'Dimension':<20} {'Avg':>6} {'Min':>6} {'Max':>6}")
print("-" * 38)
for dim in extended_dims:
    vals = [getattr(r["score"], dim) for r in concise_scored]
    print(f"{dim.capitalize():<20} {sum(vals)/len(vals):>6.2f} {min(vals):>6} {max(vals):>6}")

### Exercise 2 Answer — Break Faithfulness

**Expected result:** With `k=0`, the RAG pipeline generates answers with an empty context string. The model will either:
- Make up plausible-sounding facts (hallucinate) → Faithfulness = 0
- Say "I don't have that information" → Faithfulness may score 2-3 (not unfaithful, just empty)

Relevance may or may not drop — if the model still produces on-topic text from its training data, Relevance can stay high even though Faithfulness collapses. **This is the key insight:** a model that sounds relevant but is not grounded is a production risk.

### Exercise 3 Answer — Compare Two Models

**What to look for:** Stronger models typically improve Completeness (they cover more key points) without sacrificing Faithfulness. Weaker models may score lower on Completeness (terse or missing details) but keep Faithfulness high if they stick to the context. An unexpected result — a stronger model scores *lower* on Faithfulness — indicates it is over-generating beyond what the context supports.